# 01 — Data exploration and leakage audit

**Prerequisites** (from `ml/`):
```bash
python -m leafdx.data.download --data-dir data   # needs a Kaggle API token for NPD
python -m leafdx.data.prepare  --config configs/data.yaml
```
Every figure below is produced by `leafdx.eda` (also runnable as `python -m leafdx.eda`), so the
notebook and `docs/figures/eda/` always agree. Numbers are read from the pipeline's output files;
nothing is typed in by hand.

In [ ]:
import json, os
from pathlib import Path

# run from ml/ regardless of where Jupyter was started
root = Path.cwd()
while not (root / "pyproject.toml").exists():
    root = root.parent
os.chdir(root)

import pandas as pd
from IPython.display import Image, Markdown, display

from leafdx import eda
from leafdx.data.prepare import DataConfig

cfg = DataConfig.from_yaml(Path(os.environ.get("LEAFDX_DATA_CONFIG", "configs/data.yaml")))
FIG = cfg.report_dir.parent / "eda"
FIG.mkdir(parents=True, exist_ok=True)
npd, pairs, pdoc = eda.load(cfg)
summary = json.loads((cfg.report_dir / "summary.json").read_text())
pd.set_option("display.max_rows", 60)
show = lambda p: display(Image(filename=str(p)))

## 1. New Plant Diseases Dataset (NPD): size and class balance

In [ ]:
n = summary["npd"]
pd.Series({
    "images": n["n_images"],
    "unreadable": n["n_unreadable"],
    "paper train": n["paper_split_counts"].get("train"),
    "paper val": n["paper_split_counts"].get("val"),
    "distinct source images": n["n_source_images"],
    "images with an upstream leaf id": n["n_with_leaf_id"],
    "distinct physical leaves": n["n_physical_leaves"],
    "byte-identical extra copies": n["n_exact_duplicate_files"],
}).to_frame("value")

In [ ]:
show(eda.fig_class_counts(npd, FIG))

In [ ]:
counts = npd.groupby("label").size()
print(f"largest/smallest class ratio: {counts.max() / counts.min():.2f}")
counts.describe().to_frame("images per class").T

## 2. Sample images

In [ ]:
show(eda.fig_sample_grid_npd(npd, cfg.data_dir, FIG, cfg.seed))

In [ ]:
show(eda.fig_sample_grid_plantdoc(pdoc, cfg.data_dir, FIG, cfg.seed))

## 3. Offline augmentation

NPD keeps the PlantVillage file name and appends a suffix per transform. The table counts each
type and where it landed in the paper's 80/20 split.

In [ ]:
show(eda.fig_aug_types(npd, FIG))
pd.read_csv(cfg.report_dir / 'aug_type_counts.csv')

In [ ]:
show(eda.fig_augmented_variants(npd, cfg.data_dir, FIG, cfg.seed))

## 4. Image statistics: lab vs field

In [ ]:
display(pd.read_csv(eda.table_image_stats(npd, pdoc, FIG)).T)
show(eda.fig_background_brightness(npd, pdoc, FIG))

## 5. Leakage audit of the paper's 80/20 split

For every validation image: does the training split contain (a) an augmented copy of the same
source photo, (b) another photo of the same physical leaf, (c) a byte-identical file, or (d) a
perceptual near-duplicate (D4-invariant pHash)?

In [ ]:
a = summary["paper_split_audit"]
pd.Series({k: v for k, v in a.items() if k != "phash"}).to_frame("paper split (val vs train)")

In [ ]:
display(Markdown((cfg.report_dir / "paper_split_leakage_thresholds.md").read_text()))
show(eda.fig_leakage_curve(npd, pairs, cfg, FIG))

### How good is pHash as a detector?

Measured on the known (original, augmented copy) pairs. D4 hashing is designed to catch the
90/180/270-degree rotations and flips; 30-degree rotations and other transforms are expected to
be missed, which is why grouping uses file names first and pHash only as a backstop.

In [ ]:
pd.read_csv(cfg.report_dir / 'phash_recall_by_aug_type.csv')

In [ ]:
per_class = pd.read_csv(cfg.report_dir / "paper_split_leakage_per_class.csv")
per_class.sort_values("frac_any", ascending=False)

## 6. Choosing the pHash grouping threshold

Pairs found **only** by pHash (different source image, no shared leaf id), sampled at each
distance. Inspect these to judge where true duplicates stop and unrelated look-alikes begin;
the threshold in `configs/data.yaml` must be justified by this figure.

In [ ]:
p = eda.fig_hash_only_pairs(npd, pairs, cfg.data_dir, FIG, cfg.seed)
if p: show(p)
src, leaf = npd["source_id"].to_numpy(), npd["leaf_id"].fillna("").to_numpy()
i, j = pairs["i"].to_numpy(), pairs["j"].to_numpy()
unexpl = pairs[(src[i] != src[j]) & ~((leaf[i] != "") & (leaf[i] == leaf[j]))]
unexpl.groupby("dist").size().rename("hash-only pairs").to_frame().T

## 7. The clean grouped split (70/15/15)

In [ ]:
pd.Series({
    "groups": n["n_groups"],
    "largest group": n["group_size"]["max"],
    "label-conflict images dropped": n["n_label_conflict_images"],
    "excluded from clean split (total)": n["n_excluded_from_clean_split"],
    **{f"clean {k}": v for k, v in n["clean_split_counts"].items()},
}).to_frame("value")

In [ ]:
pd.read_csv(cfg.report_dir / 'split_counts_per_class.csv')

In [ ]:
pd.DataFrame({s: {k: v for k, v in r.items() if k != "phash"}
              for s, r in summary["clean_split_audit"].items()})

## 8. PlantDoc field test set

In [ ]:
pd.Series(summary["plantdoc"]).to_frame("value")

In [ ]:
pd.read_csv(cfg.report_dir / 'plantdoc_classes.csv')